# 🚀 Redes convolucionales con Keras, en GPU
*Curso de Machine Learning · 🚀 Llévalo a Colab · [MiAtlas.cl](https://miatlas.cl) · Licencia [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/deed.es)*

**Objetivo:** ver por qué una red **convolucional** reconoce imágenes mejor que una red densa, mirar qué aprenden sus
filtros y medir cuánto acelera la GPU.

**Datos:** MNIST, 70.000 dígitos escritos a mano de 28×28 píxeles (Yann LeCun, Corinna Cortes y Christopher Burges).

In [ ]:
# Preparación: ¿Colab?, ¿GPU? y modo de prueba
import os
try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False
PRUEBA_RAPIDA = os.environ.get("MIATLAS_PRUEBA") == "1"   # solo para la prueba automática de MiAtlas.cl

import tensorflow as tf
import keras
GPUS = tf.config.list_physical_devices("GPU")
print("Colab:", EN_COLAB, "| TensorFlow", tf.__version__, "| GPU:", GPUS[0].name if GPUS else "ninguna")
if not GPUS:
    print("⚠ Sin GPU. En Colab: Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU.")
keras.utils.set_random_seed(0)

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train, x_test = x_train[..., None] / 255.0, x_test[..., None] / 255.0     # (n, 28, 28, 1) entre 0 y 1
if PRUEBA_RAPIDA:
    x_train, y_train, x_test, y_test = x_train[:3000], y_train[:3000], x_test[:500], y_test[:500]
print(x_train.shape, x_test.shape)

import matplotlib.pyplot as plt
fig, ejes = plt.subplots(1, 10, figsize=(12, 1.5))
for i, eje in enumerate(ejes):
    eje.imshow(x_train[i, :, :, 0], cmap="gray"); eje.set_title(y_train[i]); eje.axis("off")
plt.show()

## 1. Una red densa: mira cada píxel por separado

In [ ]:
import time
EPOCAS = 1 if PRUEBA_RAPIDA else 5

densa = keras.Sequential([
    keras.Input((28, 28, 1)), keras.layers.Flatten(),
    keras.layers.Dense(128, activation="relu"), keras.layers.Dense(10, activation="softmax"),
])
densa.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
inicio = time.perf_counter()
densa.fit(x_train, y_train, epochs=EPOCAS, batch_size=128, validation_split=0.1, verbose=2)
print(f"Red densa: {densa.evaluate(x_test, y_test, verbose=0)[1]:.2%} · {densa.count_params():,} parámetros · "
      f"{time.perf_counter() - inicio:.0f} s")

## 2. Una CNN: filtros que recorren la imagen

Una capa `Conv2D` desliza filtros de 3×3 por la imagen: el **mismo** filtro detecta un borde esté donde esté. Así
necesita menos parámetros y aprovecha que los píxeles vecinos se relacionan.

In [ ]:
cnn = keras.Sequential([
    keras.Input((28, 28, 1)),
    keras.layers.Conv2D(32, 3, activation="relu", name="conv1"), keras.layers.MaxPooling2D(),
    keras.layers.Conv2D(64, 3, activation="relu", name="conv2"), keras.layers.MaxPooling2D(),
    keras.layers.Flatten(), keras.layers.Dropout(0.3), keras.layers.Dense(10, activation="softmax"),
])
cnn.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
cnn.summary()
inicio = time.perf_counter()
historia = cnn.fit(x_train, y_train, epochs=EPOCAS, batch_size=128, validation_split=0.1, verbose=2)
print(f"CNN: {cnn.evaluate(x_test, y_test, verbose=0)[1]:.2%} · {cnn.count_params():,} parámetros · "
      f"{time.perf_counter() - inicio:.0f} s")

> **Experimenta:** anota los segundos por época con GPU. Cambia a CPU (Entorno de ejecución → Cambiar tipo) y vuelve
> a entrenar la CNN: la diferencia es mayor que con la red densa, porque las convoluciones son muchas operaciones
> iguales en paralelo.

## 3. ¿Qué miran los filtros?

In [ ]:
filtros = cnn.get_layer("conv1").get_weights()[0]          # (3, 3, 1, 32)
fig, ejes = plt.subplots(2, 8, figsize=(10, 3))
for i, eje in enumerate(ejes.flat):
    eje.imshow(filtros[:, :, 0, i], cmap="bwr"); eje.axis("off")
plt.suptitle("16 filtros de la primera capa (rojo: positivo, azul: negativo)"); plt.show()

activaciones = keras.Model(cnn.inputs, cnn.get_layer("conv1").output)(x_test[:1])
fig, ejes = plt.subplots(2, 8, figsize=(10, 3))
for i, eje in enumerate(ejes.flat):
    eje.imshow(activaciones[0, :, :, i], cmap="viridis"); eje.axis("off")
plt.suptitle(f"Lo que ve cada filtro en un {y_test[0]}"); plt.show()

## 4. Aumento de datos

Girar, desplazar o acercar un poco las imágenes durante el entrenamiento enseña a la red a no depender de la posición
exacta. Las capas de aumento solo actúan al entrenar.

In [ ]:
aumento = keras.Sequential([keras.layers.RandomRotation(0.08), keras.layers.RandomTranslation(0.1, 0.1),
                           keras.layers.RandomZoom(0.1)])
fig, ejes = plt.subplots(1, 8, figsize=(10, 1.5))
for eje in ejes:
    eje.imshow(aumento(x_train[:1], training=True)[0, :, :, 0], cmap="gray"); eje.axis("off")
plt.suptitle("El mismo dígito, aumentado"); plt.show()

## Pruébalo tú

Arma `cnn_aumentada`: la misma CNN, pero con `aumento` como primera capa después de `keras.Input`. Entrénala y guarda
su exactitud en prueba en `exactitud_aumentada`. ¿Mejora? ¿Y si pruebas con dígitos movidos?

In [ ]:
exactitud_aumentada = None   # ← tu código

In [ ]:
# Comprobación
if exactitud_aumentada is None:
    print("Aún no lo resuelves.")
elif 0 < exactitud_aumentada <= 1:
    print("✅ Exactitud con aumento de datos registrada")
else:
    print("❌ Revisa tu respuesta")

## Resumen

- Las **convoluciones** reutilizan filtros por toda la imagen: menos parámetros y mejor exactitud.
- Los filtros de la primera capa detectan bordes y trazos; las capas siguientes los combinan.
- El **aumento de datos** hace a la red más robusta; la **GPU** acelera sobre todo las convoluciones.

**Fuentes:** [Keras · Conv2D](https://keras.io/api/layers/convolution_layers/convolution2d/) · [Keras · capas de aumento](https://keras.io/api/layers/preprocessing_layers/image_augmentation/) · [Keras · MNIST](https://keras.io/api/datasets/mnist/) · [TensorFlow · uso de GPU](https://www.tensorflow.org/guide/gpu)